# Product Photo Studio

Upload one product photo and get new lifestyle scenes for ads and marketplace listings.

**How it works**
1. **Segmentation:** `rembg` (IS Net) cuts the product out of the original photo.
2. **Generation:** Stable Diffusion XL Inpainting (Hugging Face `diffusers`) generates a new scene only around the product.
3. **Product protection:** the original product pixels are pasted back on top, so logos, text and colors never change.
4. **UI:** a Gradio app shows the cutout, the generated scenes, and speed per image.

Runs on a free Colab T4 GPU. Total cost: $0.

## How to run
1. **Runtime > Change runtime type > T4 GPU > Save**
2. **Runtime > Run all**
3. The first time, the session **restarts automatically** after installing packages. A "session crashed" message is expected.
4. **Runtime > Run all again.** This time installation is skipped and the app starts.
5. Open the public link ending in **gradio.live** at the bottom.


## 1. Setup (installs packages, then restarts once)

In [ ]:
import importlib.util, subprocess, sys, os

MARKER = "/content/.studio_installed"
PACKAGES = ["diffusers", "transformers", "accelerate", "rembg", "onnxruntime", "gradio"]
CHECK = ("import numpy, PIL, torch, diffusers, transformers, accelerate, rembg, onnxruntime, gradio; "
         "from diffusers import AutoPipelineForInpainting, AutoencoderKL; "
         "from rembg import remove, new_session; print('imports ok')")

def pip_install(args):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)

def imports_work():
    r = subprocess.run([sys.executable, "-c", CHECK], capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr)[-1500:]

ok, _ = imports_work()
if os.path.exists(MARKER) and ok:
    print("Setup already done. Continuing.")
else:
    print("Installing packages (about 1 to 2 minutes)...")
    import numpy
    # Keep Colab's numpy version so nothing else breaks
    with open("/content/constraints.txt", "w") as f:
        f.write(f"numpy=={numpy.__version__}\n")
    r = pip_install([*PACKAGES, "-c", "/content/constraints.txt"])
    if r.returncode != 0:
        print("Pinned install failed, retrying without the numpy pin...")
        r = pip_install(PACKAGES)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError("pip install failed. Copy the text above and send it to me.")

    ok, out = imports_work()
    if not ok:
        print(out)
        raise RuntimeError("Packages installed but imports fail. Copy the text above and send it to me.")

    print("Note: pip warnings about numba, cudf, cuml or pytensor are harmless. This project does not use them.")
    open(MARKER, "w").write("ok")
    print("\nInstall complete. Restarting the session now.")
    print("When it restarts, click Runtime > Run all again.")
    os.kill(os.getpid(), 9)

## 2. Check GPU and load models (first time about 3 to 5 minutes)

In [ ]:
import os, time, traceback
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"

import numpy as np
import torch
from PIL import Image, ImageFilter, ImageOps
from diffusers import AutoPipelineForInpainting, AutoencoderKL
from rembg import remove, new_session

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Go to Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}, {gpu.total_memory / 1e9:.1f} GB")

MODEL_ID = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"

# fp16 safe VAE: the default SDXL VAE can produce black images in half precision
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
try:
    pipe = AutoPipelineForInpainting.from_pretrained(MODEL_ID, vae=vae, torch_dtype=torch.float16, variant="fp16")
except Exception:
    pipe = AutoPipelineForInpainting.from_pretrained(MODEL_ID, vae=vae, torch_dtype=torch.float16)
pipe = pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)

seg_session = new_session("isnet-general-use")

SIZE = 1024
OUT_DIR = "/content/outputs"
os.makedirs(OUT_DIR, exist_ok=True)
print("Models ready")

## 3. Pipeline

In [ ]:
from scipy import ndimage
from PIL import ImageChops, ImageEnhance

STYLES = {
    "Campfire at dusk": "lying flat directly on a rock next to a campfire at dusk, tent in the background, warm firelight",
    "Forest trail": "lying flat directly on a hiking backpack on a sunny forest trail, green trees, clear daylight",
    "Studio marble": "lying flat directly on a white marble surface, minimal studio, soft shadows",
    "Winter night": "lying flat directly on fresh snow in a pine forest at night, blue moonlight, clear sky with stars",
    "Garage workbench": "lying flat directly on a wooden garage workbench next to tools, warm work light",
    "Mountain summit": "lying flat directly on a granite rock at a mountain summit, clear blue sky, sunny day, peaks in the distance",
}
QUALITY = ", professional product photography, sharp detailed background, vivid colors, high contrast, crisp, realistic lighting, 8k photo"
NEGATIVE = ("blurry, out of focus, fog, haze, mist, smoke, washed out, low contrast, desaturated, grainy, low quality, "
            "distorted, deformed, text, watermark, logo, duplicate product, extra objects, extra straps, extra parts, "
            "floating, levitating, cartoon, painting")

MODES = ["Blend (product inside scene)", "Clean background (no artifacts)"]


def cut_out(image, remove_white_holes=True):
    """Step 1: remove the background, clean the mask, crop tightly around the product."""
    image = ImageOps.exif_transpose(image).convert("RGB")
    image.thumbnail((2048, 2048))
    rgba = remove(image, session=seg_session)

    rgb = np.asarray(image).astype(np.int16)
    alpha = np.asarray(rgba.getchannel("A")).copy()

    # Make the product fully solid, filling faded or see through parts inside it
    solid = ndimage.binary_fill_holes(alpha > 30)

    # Remove large pure white areas inside the product (like the inside of a headband loop)
    if remove_white_holes:
        white = (rgb.min(axis=2) > 235) & ((rgb.max(axis=2) - rgb.min(axis=2)) < 15)
        labels, n = ndimage.label(white)
        if n:
            sizes = ndimage.sum(white, labels, range(1, n + 1))
            big_ids = np.where(sizes > 0.005 * white.size)[0] + 1
            big = ndimage.binary_dilation(np.isin(labels, big_ids), iterations=2)
            solid[big] = False

    # Clean small specks and shave 1 px off the edge to remove white halos
    solid = ndimage.binary_opening(solid, iterations=1)
    solid = ndimage.binary_erosion(solid, iterations=1)
    alpha_img = Image.fromarray(np.where(solid, 255, 0).astype(np.uint8)).filter(ImageFilter.GaussianBlur(0.8))

    rgba = image.convert("RGBA")
    rgba.putalpha(alpha_img)
    bbox = alpha_img.point(lambda v: 255 if v > 20 else 0).getbbox()
    if bbox is None:
        raise ValueError("No product found in the photo. Try a photo with one clear product.")
    return rgba.crop(bbox)


def place_on_canvas(product, scale=0.6, bottom=0.88, clean=False, seed=0):
    """Step 2: put the product on a square canvas and build the inpainting mask.
    White in the mask = generate. Black = keep (product).
    The background starts as random color noise instead of flat gray, which avoids washed out gray results."""
    w, h = product.size
    k = scale * SIZE / max(w, h)
    product = product.resize((max(1, int(w * k)), max(1, int(h * k))), Image.LANCZOS)
    x = (SIZE - product.width) // 2
    y = max(0, int(SIZE * bottom) - product.height)

    rng = np.random.default_rng(seed)
    noise = rng.integers(0, 256, (SIZE // 8, SIZE // 8, 3), dtype=np.uint8)
    canvas = Image.fromarray(noise).resize((SIZE, SIZE), Image.BICUBIC)

    if clean:
        # Generate the whole background without the product, so nothing can grow out of it
        mask = Image.new("L", (SIZE, SIZE), 255)
    else:
        canvas.paste(product, (x, y), product)
        alpha = Image.new("L", (SIZE, SIZE), 0)
        alpha.paste(product.getchannel("A"), (x, y))
        keep = alpha.point(lambda v: 255 if v > 128 else 0).filter(ImageFilter.MinFilter(5))
        mask = ImageOps.invert(keep)
    return canvas, mask, product, (x, y)


def pop_background(img, contrast=1.12, color=1.15):
    """Fix washed out backgrounds: a bit more contrast and color."""
    img = ImageEnhance.Contrast(img).enhance(contrast)
    return ImageEnhance.Color(img).enhance(color)


def match_light(placed, background, pos, amount=0.6):
    """Match the product's brightness and color to the scene, so a studio lit product
    does not look too bright in a dusk or night scene."""
    if amount <= 0:
        return placed
    x, y = pos
    w, h = placed.size
    pad = max(w, h) // 2
    region = background.crop((max(0, x - pad), max(0, y - pad), min(SIZE, x + w + pad), min(SIZE, y + h + pad)))
    bg = np.asarray(region).reshape(-1, 3).astype(np.float32)
    bg_mean = bg.mean(axis=0)
    bg_bright = float(np.percentile(bg @ [0.299, 0.587, 0.114], 90))

    rgb = placed.convert("RGB")
    a = np.asarray(placed.getchannel("A")) > 128
    p = np.asarray(rgb).astype(np.float32)[a]
    p_bright = float(np.percentile(p @ [0.299, 0.587, 0.114], 90)) if len(p) else bg_bright

    # Brightness: match the product's brightest parts (white logo, highlights) to the scene's bright parts
    full = float(np.clip(bg_bright / max(p_bright, 1), 0.5, 1.1))
    factor = 1 + (full - 1) * amount
    rgb = ImageEnhance.Brightness(rgb).enhance(factor)
    # Slightly softer contrast so highlights do not glow
    rgb = ImageEnhance.Contrast(rgb).enhance(1 - 0.15 * amount)
    # Color tint: pick up a little of the scene's color (warm at sunset, blue at night)
    tint = Image.new("RGB", rgb.size, tuple(int(c) for c in bg_mean))
    rgb = Image.blend(rgb, ImageChops.multiply(rgb, tint).point(lambda v: min(255, int(v * 1.6))), 0.25 * amount)

    out = rgb.convert("RGBA")
    out.putalpha(placed.getchannel("A"))
    return out


def add_shadow(background, placed, pos, strength=0.6):
    """Soft shadows so the product sits on the surface instead of floating."""
    w, h = placed.size
    a = placed.getchannel("A")

    # 1. Wide soft shadow under the whole product
    layer = Image.new("L", background.size, 0)
    flat = a.resize((w, max(1, h // 5)))
    layer.paste(flat, (pos[0] + w // 40, pos[1] + h - flat.height // 2))
    layer = layer.filter(ImageFilter.GaussianBlur(max(6, w // 30))).point(lambda v: int(v * strength))

    # 2. Contact shadow that follows the product shape, so every edge (also inside the loop) touches the ground
    contact = Image.new("L", background.size, 0)
    contact.paste(a, (pos[0] + w // 60, pos[1] + max(3, h // 30)))
    contact = contact.filter(ImageFilter.GaussianBlur(max(4, w // 60))).point(lambda v: int(v * 0.5))

    layer = ImageChops.lighter(layer, contact)
    return Image.composite(Image.new("RGB", background.size, (0, 0, 0)), background, layer)


def run_inpaint(prompt, canvas, mask, seed, steps=30, size=SIZE):
    g = torch.Generator("cuda").manual_seed(int(seed))
    return pipe(
        prompt=prompt, negative_prompt=NEGATIVE,
        image=canvas, mask_image=mask, width=size, height=size,
        num_inference_steps=steps, guidance_scale=8.0, strength=0.99, generator=g,
    ).images[0]


def generate(image, style, custom_prompt, n_images, product_size, seed,
             remove_holes=True, shadow=True, light=0.6, mode=MODES[0]):
    if image is None:
        raise gr.Error("Upload a product photo first.")
    seed = 42 if seed is None else int(seed)
    clean = mode == MODES[1]
    scene = custom_prompt.strip() if custom_prompt and custom_prompt.strip() else STYLES[style]
    if clean:
        scene += ", empty surface in the center foreground, nothing on it"
    prompt = scene + QUALITY
    log = []
    try:
        t0 = time.time()
        product = cut_out(image, remove_white_holes=remove_holes)
        log.append(f"Segmentation: {time.time() - t0:.1f}s")

        results = []
        for i in range(int(n_images)):
            t = time.time()
            canvas, mask, placed, pos = place_on_canvas(product, scale=float(product_size), clean=clean, seed=seed + i)
            out = run_inpaint(prompt, canvas, mask, seed + i).resize((SIZE, SIZE))
            out = pop_background(out)
            lit = match_light(placed, out, pos, float(light))
            if shadow:
                out = add_shadow(out, lit, pos)
            out.paste(lit, pos, lit)  # original product shape back on top, logo never redrawn
            out.save(f"{OUT_DIR}/scene_{int(time.time())}_{i + 1}.png")
            results.append(out)
            log.append(f"Image {i + 1}: {time.time() - t:.1f}s")

        log.append(f"Total: {time.time() - t0:.1f}s | Cost: $0 (open source model, free Colab GPU)")
        return product, results, "\n".join(log)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise gr.Error("GPU ran out of memory. Try fewer images.")
    except ValueError as e:
        raise gr.Error(str(e))
    except Exception:
        torch.cuda.empty_cache()
        return None, [], "Error:\n" + traceback.format_exc()[-1500:]

## 4. Health check (about 15 seconds)

Runs the whole pipeline on a test image to make sure everything works before the app starts.

In [ ]:
from PIL import ImageDraw
test = Image.new("RGB", (800, 800), (235, 235, 235))
d = ImageDraw.Draw(test)
d.rounded_rectangle((300, 200, 500, 650), radius=40, fill=(30, 90, 160))
d.ellipse((330, 120, 470, 260), fill=(200, 60, 40))

t = time.time()
cut = cut_out(test)
canvas, mask, placed, pos = place_on_canvas(cut)
small = run_inpaint("product on a wooden table" + QUALITY, canvas.resize((512, 512)), mask.resize((512, 512)), 1, steps=6, size=512)
arr = np.asarray(small)
assert arr.mean() > 5, "Generated image is black. Send me a screenshot."
print(f"Health check passed in {time.time() - t:.1f}s. Cutout size: {cut.size}")
display(small.resize((256, 256)))

## 5. Launch the app

Open the **public URL** ending in `gradio.live` below. It stays alive while this Colab tab is open (up to 72 hours).

In [ ]:
import gradio as gr

with gr.Blocks(title="Product Photo Studio") as demo:
    gr.Markdown("# Product Photo Studio\nUpload one product photo and get new scenes for ads and listings. The product itself is never changed.")
    with gr.Row():
        with gr.Column():
            inp = gr.Image(type="pil", label="Product photo")
            style = gr.Dropdown(list(STYLES), value="Campfire at dusk", label="Scene")
            custom = gr.Textbox(label="Custom scene (optional, overrides the dropdown)",
                                placeholder="lying flat directly on a rock next to a campfire at dusk, tent in the background")
            n = gr.Slider(1, 4, value=2, step=1, label="Number of images")
            size = gr.Slider(0.3, 0.75, value=0.6, step=0.05, label="Product size in frame")
            seed = gr.Number(value=42, precision=0, label="Seed (change for new variations)")
            holes = gr.Checkbox(value=True, label="Remove white areas inside the product (for loops like headbands)")
            shadow = gr.Checkbox(value=True, label="Add shadow under the product")
            light = gr.Slider(0, 1, value=0.6, step=0.1, label="Light match (0 = original product light, 1 = fully match the scene)")
            mode = gr.Radio(MODES, value=MODES[1], label="Background mode")
            btn = gr.Button("Generate", variant="primary")
        with gr.Column():
            cut_view = gr.Image(label="Step 1: product cutout")
            gallery = gr.Gallery(label="Step 2: generated scenes", columns=2, height="auto")
            info = gr.Textbox(label="Speed and cost", lines=6)
    btn.click(generate, [inp, style, custom, n, size, seed, holes, shadow, light, mode], [cut_view, gallery, info])

try:
    demo.queue().launch(share=True, inline=True)
except Exception as e:
    print("Public link failed, starting inside Colab only:", e)
    demo.queue().launch(share=False, inline=True)

## 6. Download your results

Run this any time after generating images. It downloads all of them as a zip, ready for GitHub and screenshots.

In [ ]:
import glob, shutil
files_found = glob.glob(f"{OUT_DIR}/*.png")
if not files_found:
    print("No images yet. Generate some in the app first.")
else:
    shutil.make_archive("/content/outputs", "zip", OUT_DIR)
    from google.colab import files
    files.download("/content/outputs.zip")
    print(f"Downloading {len(files_found)} images")